In [1]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os
import splitfolders
import albumentations as A
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.utils import class_weight
import math
import time
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import glob
import shutil
import random
from pathlib import Path

# Resize Gambar

In [9]:
input_folder = 'Bounding Box'
size = (640, 640)

for kelas in os.listdir(input_folder):
    nama_kelas = os.path.join(input_folder, kelas)
    if os.path.isdir(nama_kelas):
        img_folder = os.path.join(nama_kelas, "images")
        for filename in os.listdir(img_folder):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(img_folder, filename)
                img = Image.open(img_path)
                if img.mode == "RGBA":
                    img = img.convert("RGB")
                img_resized = img.resize(size)
                img_resized.save(os.path.join(img_folder, filename))

In [10]:
SOURCE_DIR = Path("D:/Klasifikasi Kutu/Detection/Bounding Box")
OUTPUT_DIR = Path("D:/Klasifikasi Kutu/Detection/Dataset")
SPLIT_RATIO = (0.7, 0.2, 0.1)   
SEED        = 42

KELAS = {
    "Dermacentor variabilis"   : 0,
    "Ixodes scapularis"        : 1,
    "Rhipicephalus sanguineus" : 2,
}

def validasi_ratio():
    assert len(SPLIT_RATIO) == 3
    assert abs(sum(SPLIT_RATIO) - 1.0) < 1e-6, \
        f"Total SPLIT_RATIO harus = 1.0, sekarang = {sum(SPLIT_RATIO)}"
    train_r, val_r, test_r = SPLIT_RATIO
    print(f"✅ Ratio valid: Train={train_r*100:.0f}% | Val={val_r*100:.0f}% | Test={test_r*100:.0f}%")


# ─── 1. BUAT FOLDER OUTPUT ────────────────────────
def buat_folder_output():
    for split in ["train", "val", "test"]:
        os.makedirs(f"{OUTPUT_DIR}/images/{split}", exist_ok=True)
        os.makedirs(f"{OUTPUT_DIR}/labels/{split}", exist_ok=True)


# ─── 2. REMAP CLASS ID ────────────────────────────
def remap_label(src_label_path, dst_label_path, class_id):
    """
    Ganti semua class ID di file .txt YOLO
    menjadi class_id yang sesuai urutan global.
    """
    lines = open(src_label_path).readlines()
    with open(dst_label_path, "w") as f:
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            parts[0] = str(class_id)
            f.write(" ".join(parts) + "\n")


# ─── 3. SPLIT DATASET ────────────────────────────
def split_dataset():
    random.seed(SEED)
    train_r, val_r, _ = SPLIT_RATIO
    total_train = 0
    total_val   = 0
    total_test  = 0

    for kelas, class_id in KELAS.items():
        img_dir = Path(SOURCE_DIR) / kelas / "images"
        lbl_dir = Path(SOURCE_DIR) / kelas / "labels"

        # Ambil semua gambar yang punya label pasangan
        images = [
            f for f in img_dir.glob("*")
            if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
            and (lbl_dir / (f.stem + ".txt")).exists()
        ]

        if not images:
            print(f"⚠️  Tidak ada gambar berlabel di: {kelas}")
            continue

        random.shuffle(images)

        # Hitung index split dari tuple
        n         = len(images)
        train_end = int(n * train_r)
        val_end   = int(n * (train_r + val_r))

        splits = {
            "train" : images[:train_end],
            "val"   : images[train_end:val_end],
            "test"  : images[val_end:],
        }

        print(f"\n📁 {kelas} (class_id={class_id}):")
        print(f"   Total : {n} gambar")
        for split_name, imgs in splits.items():
            ratio = SPLIT_RATIO[["train","val","test"].index(split_name)]
            print(f"   {split_name:5s} : {len(imgs)} ({ratio*100:.0f}%)")

        # Copy gambar + remap label
        for split_name, imgs in splits.items():
            for img_path in imgs:
                src_lbl = lbl_dir / (img_path.stem + ".txt")
                dst_img = Path(OUTPUT_DIR) / "images" / split_name / img_path.name
                dst_lbl = Path(OUTPUT_DIR) / "labels" / split_name / (img_path.stem + ".txt")

                shutil.copy2(img_path, dst_img)
                remap_label(src_lbl, dst_lbl, class_id)

        total_train += len(splits["train"])
        total_val   += len(splits["val"])
        total_test  += len(splits["test"])

    return total_train, total_val, total_test


# ─── 4. BUAT data.yaml ───────────────────────────
def buat_yaml():
    names_str   = "\n".join([f"  {id}: {nama.replace(' ', '_')}" for nama, id in KELAS.items()])
    yaml_content = f"""path: {OUTPUT_DIR}
train: images/train
val: images/val
test: images/test

nc: {len(KELAS)}
names:
{names_str}
"""
    yaml_path = f"{OUTPUT_DIR}/data.yaml"
    with open(yaml_path, "w") as f:
        f.write(yaml_content)
    print(f"\n✅ data.yaml dibuat di: {yaml_path}")
    print(yaml_content)


# ─── MAIN ─────────────────────────────────────────
if __name__ == "__main__":
    train_r, val_r, test_r = SPLIT_RATIO
    print("=" * 50)
    print("  Split Dataset YOLOv8 - 3 Kelas Kutu")
    print(f"  {train_r*100:.0f}% Train / {val_r*100:.0f}% Val / {test_r*100:.0f}% Test")
    print("=" * 50 + "\n")

    validasi_ratio()
    buat_folder_output()
    total_train, total_val, total_test = split_dataset()
    buat_yaml()

    print(f"""
{'='*50}
  ✅ Split selesai!
  Train : {total_train} gambar
  Val   : {total_val} gambar
  Test  : {total_test} gambar
  Total : {total_train + total_val + total_test} gambar
{'='*50}
""")

  Split Dataset YOLOv8 - 3 Kelas Kutu
  70% Train / 20% Val / 10% Test

✅ Ratio valid: Train=70% | Val=20% | Test=10%

📁 Dermacentor variabilis (class_id=0):
   Total : 459 gambar
   train : 321 (70%)
   val   : 92 (20%)
   test  : 46 (10%)

📁 Ixodes scapularis (class_id=1):
   Total : 556 gambar
   train : 389 (70%)
   val   : 111 (20%)
   test  : 56 (10%)

📁 Rhipicephalus sanguineus (class_id=2):
   Total : 473 gambar
   train : 331 (70%)
   val   : 94 (20%)
   test  : 48 (10%)

✅ data.yaml dibuat di: D:\Klasifikasi Kutu\Detection\Dataset/data.yaml
path: D:\Klasifikasi Kutu\Detection\Dataset
train: images/train
val: images/val
test: images/test

nc: 3
names:
  0: Dermacentor_variabilis
  1: Ixodes_scapularis
  2: Rhipicephalus_sanguineus


  ✅ Split selesai!
  Train : 1041 gambar
  Val   : 297 gambar
  Test  : 150 gambar
  Total : 1488 gambar



# Preprocessing

In [8]:
import os
import cv2
import albumentations as A
from PIL import Image

# ========================
# CONFIG
# ========================
IMAGE_DIR = 'Dataset/images/train'
LABEL_DIR = 'Dataset/labels/train'
AUGMENT_TIMES = 5

# ========================
# AUGMENTATION PIPELINE
# ========================
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(
        limit=10, 
        p=0.5,
        border_mode=cv2.BORDER_REFLECT
    ),
    A.RandomBrightnessContrast(p=0.3),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.05,
        rotate_limit=0,
        p=0.5,
        border_mode=cv2.BORDER_REFLECT
    ),
    A.HueSaturationValue(
        hue_shift_limit=5,
        sat_shift_limit=10,
        val_shift_limit=10,
        p=0.2
    ),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.2),
    A.GaussianBlur(blur_limit=(3,3), p=0.1)
], bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels'],
    min_visibility=0.3,
    clip=True
))

# ========================
# PROCESSING
# ========================
for filename in os.listdir(IMAGE_DIR):

    if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    img_path = os.path.join(IMAGE_DIR, filename)
    label_path = os.path.join(LABEL_DIR, os.path.splitext(filename)[0] + '.txt')

    # Read image
    image = cv2.imread(img_path)
    if image is None:
        print("❌ Gagal baca:", img_path)
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Read label
    bboxes = []
    class_labels = []

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                cls, x, y, w, h = map(float, line.strip().split())
                bboxes.append([x, y, w, h])
                class_labels.append(int(cls))
    else:
        print("⚠️ Label tidak ditemukan:", label_path)
        continue

    # ========================
    # AUGMENT LOOP
    # ========================
    for i in range(AUGMENT_TIMES):

        transformed = transform(
            image=image,
            bboxes=bboxes,
            class_labels=class_labels
        )

        aug_img = transformed['image']
        aug_bboxes = transformed['bboxes']
        aug_labels = transformed['class_labels']

        # Skip kalau bbox hilang semua
        if len(aug_bboxes) == 0:
            continue

        base, ext = os.path.splitext(filename)
        new_img_name = f"{base}_aug_{i+1}{ext}"
        new_lbl_name = f"{base}_aug_{i+1}.txt"

        # Save image
        Image.fromarray(aug_img).save(os.path.join(IMAGE_DIR, new_img_name))

        # Save label
        with open(os.path.join(LABEL_DIR, new_lbl_name), 'w') as f:
            for bbox, cls in zip(aug_bboxes, aug_labels):
                f.write(f"{cls} {' '.join(map(str, bbox))}\n")

- Buat Hapus File

In [ ]:
# for filename in os.listdir(IMAGE_DIR):
#     if "_aug_" in filename:
#         base_name = os.path.splitext(filename)[0]
        
#         img_path = os.path.join(IMAGE_DIR, filename)
#         lbl_path = os.path.join(LABEL_DIR, base_name + '.txt')

#         try:
#             if os.path.exists(img_path):
#                 os.remove(img_path)
#                 print(f"Deleted image: {img_path}")

#             if os.path.exists(lbl_path):
#                 os.remove(lbl_path)
#                 print(f"Deleted label: {lbl_path}")

#         except Exception as e:
#             print(f"Error: {e}")

In [6]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  

results = model.train(
    data="D:\Agung (Gungwah)\ByeFlea_Deteksi-dan-Klasifikasi-Kutu-Anjing\Detection\Dataset\data.yaml",
    epochs=30,
    imgsz=640,
    batch=8,
    lr0=0.001,
    patience=10,
    project="runs/detect",
    name="kutu_detector"
)

<>:6: SyntaxWarning: invalid escape sequence '\A'
<>:6: SyntaxWarning: invalid escape sequence '\A'
C:\Users\User\AppData\Local\Temp\ipykernel_18492\1520983999.py:6: SyntaxWarning: invalid escape sequence '\A'
  data="D:\Agung (Gungwah)\ByeFlea_Deteksi-dan-Klasifikasi-Kutu-Anjing\Detection\Dataset\data.yaml",


Ultralytics 8.4.37  Python-3.13.13 torch-2.11.0+cpu CPU (13th Gen Intel Core(TM) i7-13700F)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Agung (Gungwah)\ByeFlea_Deteksi-dan-Klasifikasi-Kutu-Anjing\Detection\Dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=kutu_detector, nbs=64, nms=False,

In [10]:
metrics = model.val()

print("===== YOLO Metrics =====")
print(f"mAP@0.5       : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95  : {metrics.box.map:.4f}")
print(f"Precision     : {metrics.box.mp:.4f}")
print(f"Recall        : {metrics.box.mr:.4f}")

Ultralytics 8.4.37  Python-3.13.13 torch-2.11.0+cpu CPU (13th Gen Intel Core(TM) i7-13700F)
val: Fast image access  (ping: 0.00.0 ms, read: 5.92.4 MB/s, size: 22.9 KB)
val: Scanning D:\Agung (Gungwah)\ByeFlea_Deteksi-dan-Klasifikasi-Kutu-Anjing\Detection\Dataset\labels\val.cache... 297 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 297/297 8.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 1.4it/s 13.1s0.7s
                   all        297        300      0.882      0.816       0.88      0.719
Dermacentor_variabilis         92         92      0.866      0.793      0.861      0.699
     Ixodes_scapularis        111        111      0.928      0.815      0.893      0.688
Rhipicephalus_sanguineus         94         97      0.853      0.838      0.885      0.769
Speed: 0.9ms preprocess, 34.4ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to D:\Agung (Gungwah)\ByeFlea_Deteksi-dan-Klasifikas

In [13]:
model = YOLO("runs/detect/runs/detect/kutu_detector/weights/best.pt")
model.save("kutu_detector_yolov8.pt")